# Assignment 7: Incremental Data Processing using Delta Lake

## Objective
Perform incremental data processing using Delta Lake on the Sample - Superstore dataset. This notebook demonstrates loading data, basic cleaning, simulating incremental data (Updates + Inserts), applying MERGE (SCD Type 1 logic), and validation.

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from delta import *
from delta.pip_utils import configure_spark_with_delta_pip

# Configure HADOOP_HOME for Windows support (requires winutils)
os.environ["HADOOP_HOME"] = os.path.abspath("../hadoop")
os.environ["PATH"] = os.environ["PATH"] + ";" + os.path.join(os.environ["HADOOP_HOME"], "bin")

# Initialize Spark Session with Delta Lake configurations
builder = pyspark.sql.SparkSession.builder.appName("DeltaLakeAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark session successfully initialized with Delta Lake support!")

Spark session successfully initialized with Delta Lake support!


### 1. Load dataset and split for simulation
We use the Sample - Superstore dataset. We will take the first 8000 rows as the initial load, and keep the remaining rows for our incremental dataset.

In [2]:
data_path = "../data/Sample - Superstore.csv"
df_full = spark.read.csv(data_path, header=True, inferSchema=True)

print(f"Total rows in full dataset: {df_full.count()}")

# Split the data using Row ID to simulate incremental load
df_initial = df_full.filter(col("Row ID") <= 8000)
df_new_records = df_full.filter(col("Row ID") > 8000)

print(f"Initial Load Rows: {df_initial.count()}")
print(f"New Records (for Incremental): {df_new_records.count()}")

Total rows in full dataset: 9994
Initial Load Rows: 8000
New Records (for Incremental): 1994


### 2. Perform basic cleaning and write initial data to Delta Table

In [3]:
# Remove any duplicates based on Row ID
df_cleaned = df_initial.dropDuplicates(["Row ID"])

# Handle nulls: e.g., fill null Postal Code with 00000
df_cleaned = df_cleaned.fillna({"Postal Code": 0})

print("Cleaned Initial Data (showing top 5):")
df_cleaned.show(5)

# Write cleaned data to Delta table
delta_table_path = "/tmp/delta/superstore_data"
df_cleaned.write.format("delta").mode("overwrite").save(delta_table_path)
print(f"Initial data written to Delta table at {delta_table_path}")

Cleaned Initial Data (showing top 5):
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|     

### 3. Create a second dataset simulating new/incremental data
We will take a sample of existing records, change their Ship Mode, and union them with the new records.

In [4]:
# Create some updates: Take 50 rows from initial data and change Ship Mode to "Same Day"
df_updates = df_initial.limit(50).withColumn("Ship Mode", lit("Same Day"))

# Combine updates with new records to form the incremental dataset
df_incremental = df_updates.union(df_new_records)

print(f"Incremental Dataset Size (Updates + Inserts): {df_incremental.count()}")
print("Sample of Incremental Data (Updates):")
df_incremental.show(5)

Incremental Dataset Size (Updates + Inserts): 2044
Sample of Incremental Data (Updates):
+------+--------------+----------+----------+---------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+---------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016| Same Day|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      424

### 4. Apply MERGE operation to update existing and insert new records

In [5]:
from delta.tables import DeltaTable

# Access the target Delta table
deltaTable = DeltaTable.forPath(spark, delta_table_path)

# Perform the Merge (SCD Type 1 logic - Upsert)
deltaTable.alias("target").merge(
    df_incremental.alias("source"),
    "target.`Row ID` = source.`Row ID`"
).whenMatchedUpdateAll(  
    # Updates all columns if matched
).whenNotMatchedInsertAll( 
    # Inserts new records if not matched
).execute()

print("MERGE operation completed successfully.")

MERGE operation completed successfully.


### 5. Validate results (row count, duplicates)

In [6]:
df_final = spark.read.format("delta").load(delta_table_path)

# Row count
print(f"Total rows in final dataset: {df_final.count()}")

# Check for duplicates
duplicates = df_final.groupBy("Row ID").count().filter("count > 1")
print(f"Number of duplicate Row IDs: {duplicates.count()}")


Total rows in final dataset: 9994
Number of duplicate Row IDs: 0


### 6. Display final dataset and verify updates

In [7]:
print("Verify the updates for Row ID <= 5 (Ship Mode should be Same Day):")
df_final.filter(col("Row ID") <= 5).select("Row ID", "Order ID", "Ship Mode").orderBy("Row ID").show()

print("Verify inserts for Row ID > 8000:")
df_final.filter(col("Row ID") > 8000).select("Row ID", "Order ID", "Ship Mode").orderBy("Row ID").show(5)


Verify the updates for Row ID <= 5 (Ship Mode should be Same Day):
+------+--------------+---------+
|Row ID|      Order ID|Ship Mode|
+------+--------------+---------+
|     1|CA-2016-152156| Same Day|
|     2|CA-2016-152156| Same Day|
|     3|CA-2016-138688| Same Day|
|     4|US-2015-108966| Same Day|
|     5|US-2015-108966| Same Day|
+------+--------------+---------+

Verify inserts for Row ID > 8000:
+------+--------------+--------------+
|Row ID|      Order ID|     Ship Mode|
+------+--------------+--------------+
|  8001|US-2015-151407|Standard Class|
|  8002|CA-2015-110870|   First Class|
|  8003|CA-2015-110870|   First Class|
|  8004|CA-2014-143210|   First Class|
|  8005|CA-2014-143210|   First Class|
+------+--------------+--------------+
only showing top 5 rows

